# 00 · Setup and GPU facts

**Goal:** prepare the Colab T4 runtime, record exact software versions, and *measure* the GPU's real limits. Every later notebook compares its results against the numbers you measure here.

**You will learn:**
1. Why GPU timing is wrong unless you synchronize, and the difference between wall-clock time and GPU time.
2. The T4's theoretical limits (compute, memory bandwidth) and what you can actually reach.
3. The *ridge point* that decides whether a layer is compute-bound or memory-bound (the basis of the roofline model in notebook 03).
4. Why host-to-GPU copies have a fixed cost that dominates small transfers.

**Before you start:** connect to a **GPU (T4)** runtime. Run the cells top to bottom.

In [ ]:
# --- Setup: make the repo's src/ folder importable on the Colab runtime ---
# The notebook runs on Google's machine, which cannot see your local folders.
# So we clone the repo there (first run) or pull the latest changes (later runs).
REPO_URL = "https://github.com/UtharaKeerthan/incabin-tensorrt.git"  # change if your repo name differs

import os, sys, subprocess, pathlib
ROOT = pathlib.Path("/content/incabin-tensorrt")
if pathlib.Path("src/bench.py").exists():            # opened from the repo root
    ROOT = pathlib.Path(".").resolve()
elif pathlib.Path("../src/bench.py").exists():       # opened from notebooks/
    ROOT = pathlib.Path("..").resolve()
elif not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)
print("repo root:", ROOT)

## 1. Install packages

Why each package:

| Package | Purpose |
|---|---|
| `tensorrt` | NVIDIA's inference compiler and runtime (the main subject) |
| `onnx`, `onnx-graphsurgeon` | Read and edit ONNX graphs |
| `onnxruntime-gpu` | A second runtime, used as a reference to check TensorRT's outputs |
| `polygraphy` | NVIDIA's tool for comparing runtimes and debugging engines |
| `nvidia-ml-py` | `pynvml`: GPU utilization, memory and clock readings |
| `nvtx` | Named ranges that show up in Nsight Systems timelines |
| `ai-edge-litert` | Google's TFLite interpreter (runs the original MediaPipe models) |
| `tf2onnx` | TFLite → ONNX converter |

`tf2onnx` is installed with `--no-deps` on purpose: its package metadata pins an old `protobuf` version, and letting pip downgrade protobuf breaks Colab's preinstalled TensorFlow.

**Jetson note:** JetPack 6 ships TensorRT 10.3. If you later move to a Jetson Orin, pin the same major version here (for example `tensorrt==10.3.*`) so your code behaves the same on both.

In [ ]:
TRT_PIN = ""   # e.g. "==10.3.*" to match JetPack 6; empty = latest
%pip install -q "tensorrt{TRT_PIN}" onnx onnxruntime-gpu onnx-graphsurgeon polygraphy nvidia-ml-py nvtx ai-edge-litert
%pip install -q --no-deps tf2onnx

## 2. What GPU did we get?

Colab usually assigns a T4, but not always. Check before trusting any number.

In [ ]:
!nvidia-smi

In [ ]:
import torch, json, platform
assert torch.cuda.is_available(), "No GPU: switch the runtime type to T4 GPU"

p = torch.cuda.get_device_properties(0)
gpu = {
    "name": p.name,
    "compute_capability": f"{p.major}.{p.minor}",   # 7.5 = Turing
    "SMs": p.multi_processor_count,                  # streaming multiprocessors
    "memory_GB": round(p.total_memory / 1e9, 2),
    "L2_cache_MB": round(getattr(p, "L2_cache_size", 0) / 2**20, 2),
}
gpu

**Read the output:**
- **Compute capability 7.5** means Turing. It decides which features exist: Turing has FP16 and INT8 Tensor Cores, but *no* TF32, *no* BF16 and *no* 2:4 sparsity (those arrived with Ampere, which is what Jetson Orin uses).
- **40 SMs**: the GPU's parallel processors. A kernel that launches fewer than 40 blocks leaves part of the GPU idle. Tiny models at batch 1 often do exactly that.
- **L2 cache (4 MB)**: the last cache before main memory. Small models' intermediate data can fit here, which makes them faster than a DRAM-bandwidth calculation predicts.

In [ ]:
import importlib, sys

def ver(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception as e:
        return f"not installed ({type(e).__name__})"

env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda (torch build)": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tensorrt": ver("tensorrt"),
    "onnx": ver("onnx"),
    "onnxruntime": ver("onnxruntime"),
    "tf2onnx": ver("tf2onnx"),
    "driver": __import__("subprocess").run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.strip(),
    **gpu,
}
json.dump(env, open("results/env.json", "w"), indent=2)
env

These versions go in your README. A TensorRT engine only runs on the TensorRT version **and** GPU that built it, so results without versions are not reproducible.

---
## 3. Theory: spec-sheet limits and the ridge point

Every operation needs two things: **arithmetic** (FLOPs) and **data movement** (bytes from memory). A GPU has a ceiling for each:

- Peak compute: how many FLOPs per second the cores can do.
- Peak bandwidth: how many bytes per second memory can deliver.

**Arithmetic intensity (AI)** of an operation = FLOPs ÷ bytes moved.

**Ridge point** = peak compute ÷ peak bandwidth. If an operation's AI is *below* the ridge point, memory cannot feed the cores fast enough: the operation is **memory-bound**. *Above* it: **compute-bound**.

The ridge point depends on precision, because Tensor Cores make FP16 and INT8 math much faster while memory bandwidth stays the same.

In [ ]:
import pandas as pd

# NVIDIA T4 datasheet values (boost clock, dense math)
T4_SPEC = {
    "FP32 (CUDA cores)":       8.1e12,
    "FP16 (Tensor Cores)":     65e12,
    "INT8 (Tensor Cores)":     130e12,
}
T4_BW = 320e9          # bytes/s, GDDR6
T4_PCIE = 15.75e9      # bytes/s, PCIe Gen3 x16 theoretical per direction

rows = [{"precision": k, "peak_TFLOPS_or_TOPS": v / 1e12,
         "ridge_point_FLOP_per_byte": round(v / T4_BW, 1)} for k, v in T4_SPEC.items()]
pd.DataFrame(rows)

**What the ridge points mean in practice:** a 3×3 depthwise convolution (common in MediaPipe and MobileNet models) has an arithmetic intensity of roughly 2–5 FLOP/byte. That is far below even the FP32 ridge point of ~25. So on a T4, such layers are memory-bound at every precision: FP16 helps them mainly by **halving the bytes**, not by faster math.

Spec sheets are upper limits. Next we measure what we can actually reach.

---
## 4. Experiment A: why GPU timing needs a sync

**Concept:** GPU calls are *asynchronous*. `c = a @ b` returns as soon as the work is queued, often in microseconds, while the GPU may still need many milliseconds to finish. If you read the clock without waiting, you measure the queueing, not the work.

`torch.cuda.synchronize()` blocks the CPU until all queued GPU work is done.

**✍️ Prediction (write it down before running):** a 4096×4096 FP32 matrix multiply is 2·4096³ ≈ 137 GFLOP. At ~8 TFLOPS that should take about ____ ms. Without a sync, the timer will show about ____ ms.

In [ ]:
import time
from src import bench

n = 4096
a = torch.randn(n, n, device="cuda")
b = torch.randn(n, n, device="cuda")
for _ in range(5):
    a @ b
torch.cuda.synchronize()

t0 = time.perf_counter(); c = a @ b; t_nosync = (time.perf_counter() - t0) * 1e3
torch.cuda.synchronize()
t0 = time.perf_counter(); c = a @ b; torch.cuda.synchronize(); t_sync = (time.perf_counter() - t0) * 1e3

print(f"one call, no sync : {t_nosync:8.3f} ms   <- only the time to queue the work")
print(f"one call, synced  : {t_sync:8.3f} ms   <- the real time")

Now the same comparison with the shared harness, over many runs. Three methods:

1. **No sync** (wrong for GPU work).
2. **Wall-clock + sync**: what an application experiences.
3. **CUDA events**: timestamps recorded on the GPU itself; GPU execution time only.

Watch the **no-sync** result closely. The GPU's launch queue has a limited size. Once it fills up, new calls must wait for old work to finish, so later "no sync" samples suddenly become slow. The distribution splits into fast (queue not full) and slow (queue full) samples. That's why the p50 and p99 differ wildly.

In [ ]:
mm = lambda: a @ b
r_nosync = bench.benchmark(mm, name="no sync (wrong)", warmup=5, iters=200, sync=None)
r_wall   = bench.benchmark(mm, name="wall-clock + sync", warmup=5, iters=50)
r_event  = bench.benchmark_cuda_events(mm, name="CUDA events", warmup=5, iters=50)
torch.cuda.synchronize()

res_a = [r_nosync, r_wall, r_event]
display(bench.results_table(res_a)[["name", "p50_ms", "p99_ms", "min_ms", "max_ms"]])
bench.plot_latency(res_a, title="Same matmul, three timing methods", log=True);

**Interpretation:**
- Wall-clock and CUDA events should agree closely here, because the matmul is long (~tens of ms) and the fixed overhead (~10–20 µs) is tiny in comparison.
- For a **tiny** model that runs in 0.1 ms, that same overhead is a large fraction of the total, and the two methods will disagree a lot. You'll see this in notebook 01 and fix it in notebook 05 (CUDA Graphs).

| Method | Measures | Pros | Cons |
|---|---|---|---|
| perf_counter, no sync | Queue time | None for GPU work | Wrong answer |
| perf_counter + sync | End-to-end, as the app sees it | Includes everything real: Python, launches, copies | Sync itself adds a little overhead; includes noise from the CPU side |
| CUDA events | GPU execution only | Precise, excludes Python | Hides CPU-side costs the app still pays |
| Nsight Systems | Full timeline of CPU and GPU | Shows *where* time goes, gaps between kernels | Heavier; a profiler, not a benchmark |

---
## 5. Experiment B: measure the real ceilings

We measure the three numbers that the roofline model needs:
1. **Achievable memory bandwidth**: copy a large buffer on the GPU. A copy reads N bytes and writes N bytes, so it moves 2N bytes.
2. **Achievable FP32 compute**: large matmul in FP32.
3. **Achievable FP16 compute**: large matmul in FP16 (uses Tensor Cores).

**✍️ Prediction:** what fraction of the spec sheet will we reach? (Common answer: 70–90% for bandwidth; FP16 is often well below 65 TFLOPS on a T4 because the 70 W power limit lowers the clock under heavy load.)

In [ ]:
def gpu_ms(fn, iters=30):
    return bench.benchmark_cuda_events(fn, name="tmp", warmup=5, iters=iters).p50

# 1) Bandwidth: device-to-device copy of 512 MB
N = 512 * 2**20
src = torch.empty(N, dtype=torch.uint8, device="cuda")
dst = torch.empty_like(src)
t = gpu_ms(lambda: dst.copy_(src))
bw = 2 * N / (t / 1e3)

# 2) FP32 matmul
n = 4096
a32 = torch.randn(n, n, device="cuda"); b32 = torch.randn(n, n, device="cuda")
t = gpu_ms(lambda: a32 @ b32, iters=20)
fp32 = 2 * n**3 / (t / 1e3)

# 3) FP16 matmul (Tensor Cores)
n16 = 8192
a16 = torch.randn(n16, n16, device="cuda", dtype=torch.float16); b16 = torch.randn_like(a16)
t = gpu_ms(lambda: a16 @ b16, iters=20)
fp16 = 2 * n16**3 / (t / 1e3)

# 4) INT8 matmul (Tensor Cores). torch._int_mm is a low-level op; may not exist in every build.
try:
    a8 = torch.randint(-64, 64, (n16, n16), device="cuda", dtype=torch.int8)
    b8 = torch.randint(-64, 64, (n16, n16), device="cuda", dtype=torch.int8)
    t = gpu_ms(lambda: torch._int_mm(a8, b8), iters=20)
    int8 = 2 * n16**3 / (t / 1e3)
except Exception as e:
    print("INT8 matmul not measured:", type(e).__name__, e)
    int8 = None

del src, dst, a32, b32, a16, b16
torch.cuda.empty_cache()

meas = pd.DataFrame([
    {"metric": "memory bandwidth (GB/s)", "spec": T4_BW / 1e9, "measured": bw / 1e9},
    {"metric": "FP32 compute (TFLOPS)", "spec": 8.1, "measured": fp32 / 1e12},
    {"metric": "FP16 compute (TFLOPS)", "spec": 65, "measured": fp16 / 1e12},
    {"metric": "INT8 compute (TOPS)", "spec": 130, "measured": (int8 or float("nan")) / 1e12},
])
meas["% of spec"] = (100 * meas["measured"] / meas["spec"]).round(1)
meas.round(1)

In [ ]:
# Ridge points from *measured* ceilings: these are the ones to use in notebook 03.
ceil = {"bw_Bps": bw, "fp32_flops": fp32, "fp16_flops": fp16, "int8_ops": int8}
ridge = {k: round(v / bw, 1) for k, v in [("fp32", fp32), ("fp16", fp16), ("int8", int8)] if v}
ceil["ridge_points"] = ridge
json.dump(ceil, open("results/t4_ceilings.json", "w"), indent=2)
print("measured ridge points (FLOP/byte):", ridge)

In [ ]:
# Clocks and power right after the heavy work: is the GPU power-limited?
!nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu --format=csv

**Interpretation:**
- If the SM clock is well below its maximum while power draw sits near the 70 W limit, the GPU is **power-throttling**. That explains a low FP16 result. Throttling also makes benchmarks drift over time, which is why we always report medians and p99 rather than single runs.
- The same effect is much stronger on a Jetson, where you choose a power mode (for example 15 W or 25 W) and the board is passively cooled.

---
## 6. Experiment C: the cost of copying data to the GPU

**Concept:** CPU memory and T4 memory are separate, connected by PCIe Gen3 x16 (~15.75 GB/s theoretical, ~12 GB/s in practice). Each copy has:
- a **fixed cost** (launch and setup, around 10 µs), and
- a **per-byte cost** (size ÷ bandwidth).

Small copies are dominated by the fixed cost, so their effective bandwidth is terrible. This is the same idea as the roofline: a "latency-bound" regime and a "bandwidth-bound" regime.

There's a second effect: **pageable vs pinned memory.** Normal (pageable) CPU memory can be moved around by the operating system, so the driver first copies it into a hidden pinned buffer and then sends it over PCIe. Pinned (page-locked) memory skips that extra copy and allows asynchronous transfers.

**✍️ Prediction:** at what transfer size does effective bandwidth reach half of its maximum?

In [ ]:
import numpy as np, matplotlib.pyplot as plt

sizes = [2**k for k in range(10, 27)]          # 1 KB ... 64 MB
rows = []
for s in sizes:
    host_pageable = torch.empty(s, dtype=torch.uint8)
    host_pinned = torch.empty(s, dtype=torch.uint8).pin_memory()
    dev = torch.empty(s, dtype=torch.uint8, device="cuda")
    for kind, h in [("pageable", host_pageable), ("pinned", host_pinned)]:
        r = bench.benchmark(lambda: dev.copy_(h, non_blocking=True), name=kind, warmup=10, iters=100)
        rows.append({"bytes": s, "kind": kind, "p50_ms": r.p50, "GB_per_s": s / (r.p50 / 1e3) / 1e9})
h2d = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
for kind, g in h2d.groupby("kind"):
    ax.plot(g["bytes"], g["GB_per_s"], marker="o", label=kind)
ax.axhline(T4_PCIE / 1e9, ls="--", c="gray", label="PCIe Gen3 x16 theoretical")
ax.set_xscale("log", base=2); ax.set_xlabel("transfer size (bytes)"); ax.set_ylabel("effective GB/s")
ax.set_title("Host → GPU copy: latency-bound for small sizes, bandwidth-bound for large")
ax.legend(); ax.grid(alpha=0.3)

In [ ]:
# The sizes that matter for our in-cabin pipeline
cases = {
    "1080p RGB frame, uint8": 1920 * 1080 * 3,
    "1080p RGB frame, float32": 1920 * 1080 * 3 * 4,
    "256x256 face crop, uint8": 256 * 256 * 3,
    "256x256 face crop, float32": 256 * 256 * 3 * 4,
    "478 landmarks x 3, float32 (output)": 478 * 3 * 4,
}
rows = []
for label, s in cases.items():
    h = torch.empty(s, dtype=torch.uint8).pin_memory()
    d = torch.empty(s, dtype=torch.uint8, device="cuda")
    r = bench.benchmark(lambda: d.copy_(h, non_blocking=True), name=label, warmup=10, iters=200)
    rows.append({"data": label, "MB": round(s / 1e6, 3), "pinned_p50_ms": round(r.p50, 4)})
pd.DataFrame(rows)

**Interpretation, and why this shapes the whole project:**
- Sending a full 1080p frame as **float32** costs about 4× more than sending it as uint8. Rule: convert to float *on the GPU*, not before the copy. (Notebook 06 does this, and notebook 07 bakes the conversion into the model.)
- A face crop is small enough that its copy is close to the fixed-cost floor. Cropping on the CPU and sending only the crop is cheap; sending the whole frame and cropping on the GPU costs more transfer but frees the CPU. That trade-off is measured in notebook 06.
- The landmark *output* copy back to the CPU costs almost only the fixed overhead.
- **Jetson:** CPU and GPU share one memory, so these copies can disappear entirely (zero-copy). This is one of the biggest differences between a T4 and an Orin.

---
## Summary

| Measured here | Used in |
|---|---|
| Software versions (`results/env.json`) | README, every notebook |
| Real bandwidth, FP32/FP16/INT8 ceilings, ridge points (`results/t4_ceilings.json`) | 03 roofline |
| Timing method behaviour (sync, CUDA events) | Every benchmark |
| Copy cost vs size, pinned vs pageable | 06 pipeline |

## Exercises
1. Repeat Experiment B three times in a row. How much do the results drift? Check `nvidia-smi` clocks between runs.
2. In Experiment A, time a *small* matmul (n = 256) with wall-clock + sync and with CUDA events. Explain the gap.
3. Add GPU → host copies to Experiment C. Are they symmetric with host → GPU?
4. In Experiment B, try n = 2048 and n = 1024 for the FP16 matmul. Why do the achieved TFLOPS drop for smaller sizes? (Hint: think about the 40 SMs and arithmetic intensity of a matmul, which is roughly n/3 FLOP/byte in FP16.)